# 🏥 Multi-Disease Prediction System with Model Comparison
> Predicts **Diabetes** and **Heart Disease** using multiple ML algorithms with full comparison dashboard

**Features:**
- ✅ Multiple diseases (Diabetes + Heart Disease)
- ✅ Multiple algorithms (LR, RF, SVM, KNN, XGBoost)
- ✅ Handles imbalanced data with SMOTE
- ✅ Feature selection (SelectKBest)
- ✅ Full metrics comparison (Accuracy, Precision, Recall, F1, ROC-AUC)
- ✅ Visualization dashboard (confusion matrix, ROC curves, feature importance)

## 📦 Step 1: Install & Import Libraries

In [ ]:
# Install required libraries
!pip install imbalanced-learn xgboost --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              roc_curve, classification_report)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# Imbalanced data
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

print('✅ All libraries imported successfully!')

## 📊 Step 2: Load Datasets
We use two well-known medical datasets:
- **Diabetes**: Pima Indians Diabetes (from UCI via URL)
- **Heart Disease**: Cleveland Heart Disease (from UCI via URL)

In [ ]:
# ── Load Diabetes Dataset ──────────────────────────────────────────────────────
diabetes_url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
diabetes_cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
                 'Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']
diabetes_df = pd.read_csv(diabetes_url, names=diabetes_cols)

# ── Load Heart Disease Dataset ─────────────────────────────────────────────────
heart_url = 'https://raw.githubusercontent.com/rohan-paul/MachineLearning-DeepLearning-Code-for-my-YouTube-Channel/master/Machine-Learning-with-Python/Heart_Disease_UCI_Dataset/heart.csv'
heart_df = pd.read_csv(heart_url)
# Rename target column to 'Outcome' for consistency
heart_df = heart_df.rename(columns={'target': 'Outcome'})

print('📋 Diabetes Dataset:', diabetes_df.shape)
print('   Class balance:\n', diabetes_df['Outcome'].value_counts().to_string())
print()
print('💓 Heart Disease Dataset:', heart_df.shape)
print('   Class balance:\n', heart_df['Outcome'].value_counts().to_string())

In [ ]:
# Quick preview of both datasets
print('=== Diabetes Dataset (first 3 rows) ===')
display(diabetes_df.head(3))

print('\n=== Heart Disease Dataset (first 3 rows) ===')
display(heart_df.head(3))

## 🔍 Step 3: EDA — Class Distribution & Correlation

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('Exploratory Data Analysis', fontsize=16, fontweight='bold')

# Class distributions
for ax, (df, title, colors) in zip(
    axes[:2],
    [(diabetes_df, 'Diabetes', ['#2ecc71','#e74c3c']),
     (heart_df,   'Heart Disease', ['#3498db','#e74c3c'])]
):
    counts = df['Outcome'].value_counts()
    ax.bar(['Negative','Positive'], counts.values, color=colors, edgecolor='black', linewidth=0.8)
    ax.set_title(f'{title} — Class Balance', fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 3, str(v), ha='center', fontweight='bold')

# Correlation heatmaps (top 6 features only for readability)
for ax, (df, title) in zip(
    axes[2:],
    [(diabetes_df, 'Diabetes Correlation'),
     (heart_df,   'Heart Disease Correlation')]
):
    top_features = df.corr()['Outcome'].abs().nlargest(6).index
    corr_matrix = df[top_features].corr()
    sns.heatmap(corr_matrix, ax=ax, annot=True, fmt='.2f', cmap='coolwarm',
                linewidths=0.5, annot_kws={'size': 7})
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## ⚙️ Step 4: Core ML Engine
Handles preprocessing, SMOTE, feature selection, training, and evaluation in one reusable function.

In [ ]:
# ── Define all models ──────────────────────────────────────────────────────────
MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM':                 SVC(probability=True, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'XGBoost':             XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                                         random_state=42, verbosity=0),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}


def train_and_evaluate(df, disease_name, use_smote=True, k_best_features=8):
    """
    Full ML pipeline for a given disease dataset.
    Returns a results DataFrame and preprocessed data for visualization.
    """
    print(f'\n{'='*60}')
    print(f'  🔬 Processing: {disease_name}')
    print(f'{'='*60}')

    # ── Split features / target ────────────────────────────────────────────────
    X = df.drop('Outcome', axis=1)
    y = df['Outcome']

    # ── Feature Selection ──────────────────────────────────────────────────────
    k = min(k_best_features, X.shape[1])
    selector = SelectKBest(f_classif, k=k)
    X_selected = selector.fit_transform(X, y)
    selected_feature_names = X.columns[selector.get_support()].tolist()
    print(f'  📌 Top {k} features selected: {selected_feature_names}')

    # ── Train / Test Split ─────────────────────────────────────────────────────
    X_train, X_test, y_train, y_test = train_test_split(
        X_selected, y, test_size=0.2, random_state=42, stratify=y
    )

    # ── Scaling ────────────────────────────────────────────────────────────────
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    # ── Handle Imbalanced Data with SMOTE ─────────────────────────────────────
    if use_smote:
        smote = SMOTE(random_state=42)
        X_train, y_train = smote.fit_resample(X_train, y_train)
        print(f'  ⚖️  After SMOTE — Class counts: {dict(zip(*np.unique(y_train, return_counts=True)))}')

    # ── Train & Evaluate Each Model ────────────────────────────────────────────
    results = []
    trained_models = {}

    for name, model in MODELS.items():
        model.fit(X_train, y_train)
        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        results.append({
            'Model':     name,
            'Accuracy':  round(accuracy_score(y_test,  y_pred),  4),
            'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
            'Recall':    round(recall_score(y_test,    y_pred),  4),
            'F1 Score':  round(f1_score(y_test,        y_pred),  4),
            'ROC-AUC':   round(roc_auc_score(y_test,   y_proba), 4),
        })
        trained_models[name] = (model, y_proba)

    results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False)
    results_df = results_df.reset_index(drop=True)
    results_df.index += 1   # rank from 1

    print(f'\n  📊 Results Summary:')
    display(results_df.style.background_gradient(cmap='YlGn', subset=['Accuracy','F1 Score','ROC-AUC']))

    return results_df, trained_models, X_test, y_test, selected_feature_names


print('✅ Engine ready!')

## 🚀 Step 5: Run Predictions on Both Diseases

In [ ]:
# Run for Diabetes
diabetes_results, diabetes_models, X_test_d, y_test_d, feats_d = train_and_evaluate(
    diabetes_df, 'Diabetes', use_smote=True, k_best_features=8
)

# Run for Heart Disease
heart_results, heart_models, X_test_h, y_test_h, feats_h = train_and_evaluate(
    heart_df, 'Heart Disease', use_smote=True, k_best_features=8
)

## 📈 Step 6: Visualization Dashboard

In [ ]:
# ── Plot 1: Grouped Metric Bar Chart ──────────────────────────────────────────
def plot_metrics_comparison(results_df, title):
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
    x = np.arange(len(results_df))
    width = 0.15
    colors = ['#3498db','#2ecc71','#e67e22','#e74c3c','#9b59b6']

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, (metric, color) in enumerate(zip(metrics, colors)):
        bars = ax.bar(x + i*width, results_df[metric], width,
                      label=metric, color=color, alpha=0.85, edgecolor='white')
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{bar.get_height():.2f}', ha='center', va='bottom',
                    fontsize=7.5, rotation=90)

    ax.set_xticks(x + width*2)
    ax.set_xticklabels(results_df['Model'], rotation=15, ha='right', fontsize=10)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    ax.set_title(f'{title} — Model Comparison', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.show()

plot_metrics_comparison(diabetes_results, '🩺 Diabetes')
plot_metrics_comparison(heart_results,    '💓 Heart Disease')

In [ ]:
# ── Plot 2: ROC Curves ────────────────────────────────────────────────────────
def plot_roc_curves(models_dict, X_test, y_test, title):
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = plt.cm.tab10(np.linspace(0, 1, len(models_dict)))

    for (name, (model, y_proba)), color in zip(models_dict.items(), colors):
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc = roc_auc_score(y_test, y_proba)
        ax.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={auc:.3f})')

    ax.plot([0,1],[0,1],'k--', lw=1.5, label='Random Classifier')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'{title} — ROC Curves', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_roc_curves(diabetes_models, X_test_d, y_test_d, '🩺 Diabetes')
plot_roc_curves(heart_models,    X_test_h, y_test_h, '💓 Heart Disease')

In [ ]:
# ── Plot 3: Confusion Matrices for Best Model ─────────────────────────────────
def plot_confusion_matrix(models_dict, X_test, y_test, title):
    # Pick the best model (first in sorted results = highest F1)
    best_name = list(models_dict.keys())[0]
    best_model, _ = models_dict[best_name]
    y_pred = best_model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative','Positive'],
                yticklabels=['Negative','Positive'],
                linewidths=1, ax=ax)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('Actual',    fontsize=11)
    ax.set_title(f'{title}\nBest Model: {best_name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f'\n📋 Classification Report — {title} [{best_name}]')
    print(classification_report(y_test, y_pred, target_names=['Negative','Positive']))

plot_confusion_matrix(diabetes_models, X_test_d, y_test_d, '🩺 Diabetes')
plot_confusion_matrix(heart_models,    X_test_h, y_test_h, '💓 Heart Disease')

In [ ]:
# ── Plot 4: Feature Importance (Random Forest) ────────────────────────────────
def plot_feature_importance(models_dict, feature_names, title):
    rf_model, _ = models_dict['Random Forest']
    importances = rf_model.feature_importances_
    indices = np.argsort(importances)[::-1]

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(range(len(importances)),
                  importances[indices],
                  color=plt.cm.viridis(np.linspace(0.2, 0.8, len(importances))),
                  edgecolor='white')
    ax.set_xticks(range(len(importances)))
    ax.set_xticklabels([feature_names[i] for i in indices], rotation=30, ha='right')
    ax.set_title(f'{title} — Feature Importance (Random Forest)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Importance Score')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    for bar, val in zip(bars, importances[indices]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8.5)
    plt.tight_layout()
    plt.show()

plot_feature_importance(diabetes_models, feats_d, '🩺 Diabetes')
plot_feature_importance(heart_models,    feats_h, '💓 Heart Disease')

## 🏆 Step 7: Final Summary — Best Model per Disease

In [ ]:
print('='*65)
print('  🏆  FINAL RESULTS SUMMARY')
print('='*65)

for disease, results in [('🩺 Diabetes', diabetes_results), ('💓 Heart Disease', heart_results)]:
    best = results.iloc[0]
    print(f'\n  {disease}')
    print(f'  Best Model  : {best["Model"]}')
    print(f'  Accuracy    : {best["Accuracy"]:.2%}')
    print(f'  F1 Score    : {best["F1 Score"]:.2%}')
    print(f'  ROC-AUC     : {best["ROC-AUC"]:.4f}')

print('\n' + '='*65)

# Side-by-side comparison heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (title, df) in zip(axes, [('Diabetes', diabetes_results), ('Heart Disease', heart_results)]):
    metric_data = df.set_index('Model')[['Accuracy','Precision','Recall','F1 Score','ROC-AUC']]
    sns.heatmap(metric_data, annot=True, fmt='.3f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, vmin=0.5, vmax=1.0,
                annot_kws={'size': 10})
    ax.set_title(f'{title} — All Models Heatmap', fontweight='bold', fontsize=12)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('📊 Complete Model Comparison Dashboard', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🩺 Step 8: Predict on New Patient Data

In [ ]:
# ── Predict for a new patient ─────────────────────────────────────────────────
# Retrain best model pipeline for diabetes on full dataset for demo
from sklearn.pipeline import Pipeline

def predict_new_patient(df, feature_cols, patient_data, disease_name):
    """
    Train best model (Random Forest) and predict for a new patient.
    patient_data: list of feature values in same order as feature_cols
    """
    X = df[feature_cols]
    y = df['Outcome']

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_scaled, y)

    patient_df = pd.DataFrame([patient_data], columns=feature_cols)
    patient_scaled = scaler.transform(patient_df)

    pred  = model.predict(patient_scaled)[0]
    proba = model.predict_proba(patient_scaled)[0]

    result = '⚠️  POSITIVE (High Risk)' if pred == 1 else '✅ NEGATIVE (Low Risk)'
    print(f'\n🔍 {disease_name} Prediction for New Patient')
    print(f'   Input features: {dict(zip(feature_cols, patient_data))}')
    print(f'   Prediction    : {result}')
    print(f'   Confidence    : Negative={proba[0]:.2%}  |  Positive={proba[1]:.2%}')


# Example diabetes patient
diabetes_features = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
                     'Insulin','BMI','DiabetesPedigreeFunction','Age']
new_diabetes_patient = [2, 148, 72, 35, 0, 33.6, 0.627, 50]  # known positive case
predict_new_patient(diabetes_df, diabetes_features, new_diabetes_patient, 'Diabetes')

# Example heart disease patient
heart_features = ['age','sex','cp','trestbps','chol','fbs','restecg',
                  'thalach','exang','oldpeak','slope','ca','thal']
new_heart_patient = [63, 1, 3, 145, 233, 1, 0, 150, 0, 2.3, 0, 0, 1]
predict_new_patient(heart_df, heart_features, new_heart_patient, 'Heart Disease')

---
## ✅ What This Project Covers

| Feature | Implementation |
|---|---|
| Multiple diseases | Diabetes + Heart Disease |
| Multiple algorithms | LR, RF, SVM, KNN, XGBoost, GradBoost |
| Imbalanced data | SMOTE oversampling |
| Feature selection | SelectKBest (f_classif) |
| Metrics | Accuracy, Precision, Recall, F1, ROC-AUC |
| Visualization | Bar charts, ROC curves, Confusion matrix, Feature importance, Heatmap |
| New prediction | Single patient inference function |